# 01 - EDA & Data Cleaning
Cleaning script for `used_car_dataset.csv` (Member 4).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

raw_candidates = [
    Path("data/used_car_dataset.csv"),
    Path("used_car_dataset.csv"),
    Path("Data-Mining-Project/regression/data/used_car_dataset.csv")
]
raw_path = next((p for p in raw_candidates if p.exists()), Path("used_car_dataset.csv"))

df = pd.read_csv(raw_path)
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after initial deduplication:", df.shape)

In [ ]:
df["kmDriven"] = pd.to_numeric(
    df["kmDriven"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)

df["AskPrice"] = pd.to_numeric(
    df["AskPrice"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)

df[["kmDriven", "AskPrice"]].dtypes

In [ ]:
num_cols = ["Year", "Age", "kmDriven", "AskPrice"]
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"Outliers detected in {col}: {outliers_count} (preserved)")

out_path = Path("data/cleaned_used_car_dataset.csv") if Path("data").exists() else Path("cleaned_used_car_dataset.csv")
df.to_csv(out_path, index=False)
print("Done! Saved cleaned_used_car_dataset.csv with shape:", df.shape)